In [7]:
import pandas as pd

test = pd.read_csv('https://raw.githubusercontent.com/vkoul/data/refs/heads/main/misc/spanish_test_table.csv')
users = pd.read_csv('https://raw.githubusercontent.com/vkoul/data/refs/heads/main/misc/spanish_user_table.csv')

In [8]:
print(test.shape)
print(users.shape)

(453321, 9)
(452867, 4)


In [9]:
print(test.head())
print(users.head())

   user_id        date  source  device browser_language ads_channel  \
0   315281  2015-12-03  Direct     Web               ES         NaN   
1   497851  2015-12-04     Ads     Web               ES      Google   
2   848402  2015-12-04     Ads     Web               ES    Facebook   
3   290051  2015-12-03     Ads  Mobile            Other    Facebook   
4   548435  2015-11-30     Ads     Web               ES      Google   

       browser  conversion  test  
0           IE           1     0  
1           IE           0     1  
2       Chrome           0     0  
3  Android_App           0     1  
4      FireFox           0     1  
   user_id sex  age    country
0   765821   M   20     Mexico
1   343561   F   27  Nicaragua
2   118744   M   23   Colombia
3   987753   F   27  Venezuela
4   554597   F   20      Spain


453,321 vs 452,867 - the difference of 454 rows or users , Which means this 454 users exist in test table but not in user table

In [10]:
print(test.info())
print(users.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 453321 entries, 0 to 453320
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   user_id           453321 non-null  int64 
 1   date              453321 non-null  object
 2   source            453321 non-null  object
 3   device            453321 non-null  object
 4   browser_language  453321 non-null  object
 5   ads_channel       181877 non-null  object
 6   browser           453321 non-null  object
 7   conversion        453321 non-null  int64 
 8   test              453321 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 31.1+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 452867 entries, 0 to 452866
Data columns (total 4 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   user_id  452867 non-null  int64 
 1   sex      452867 non-null  object
 2   age      452867 non-null  int64 
 3   country  452

Here **ads_channel** missing values Approx 40% data missing.


In [11]:
df = test.merge(users, on='user_id', how='inner')
print(df.shape)
df.head()

(452867, 12)


,user_id,date,source,device,browser_language,ads_channel,browser,conversion,test,sex,age,country
0,315281,2015-12-03,Direct,Web,ES,NaN,IE,1,0,M,32,Spain
1,497851,2015-12-04,Ads,Web,ES,Google,IE,0,1,M,21,Mexico
2,848402,2015-12-04,Ads,Web,ES,Facebook,Chrome,0,0,M,34,Spain
3,290051,2015-12-03,Ads,Mobile,Other,Facebook,Android_App,0,1,F,22,Mexico
4,548435,2015-11-30,Ads,Web,ES,Google,FireFox,0,1,M,19,Mexico


While joining the tables can drops those 454 users because it's only pull those which already matching the user_id in both the tables


In [12]:
#We need to confirm now if the test is actually negative ,
#i.e, the old version of the site with just one translation across spain and latAm performs better.


In [13]:
# Counts by test group
df.groupby('test').agg(
    users=('user_id', 'count'),
    converted=('conversion', 'sum'),
    conversion_rate=('conversion', 'mean')
).round(4)

,users,converted,conversion_rate
test,,,
0,237093,13077,0.0552
1,215774,9367,0.0434


In [14]:
from statsmodels.stats.proportion import proportions_ztest

# Number of conversions in each group
conversions = [13077, 9367]  # control, test

# Number of users in each group
nobs = [237093, 215774]  # control, test

# Two-sided z-test (just asking "are they different?")
stat, p_value = proportions_ztest(conversions, nobs)
print(f'z-statistic: {stat:.4f}')
print(f'p-value: {p_value}')

z-statistic: 18.1877
p-value: 6.46031758473152e-74


The 1.2 percentage point gap between control (5.52%) and test (4.34%) is highly statistically significant. With 450,000 users, this difference is virtually impossible to be random chance. The test really does show worse conversion under the new local translations.

In [15]:
#Explain why that might be happening. Are the localized translations really worse?

In [16]:
# Conversion rate by country and test group
country_breakdown = df.groupby(['country', 'test']).agg(
    users=('user_id', 'count'),
    converted=('conversion', 'sum'),
    conversion_rate=('conversion', 'mean')
).round(4)

print(country_breakdown)

                  users  converted  conversion_rate
country     test                                   
Argentina   0      9356        141           0.0151
            1     37377        513           0.0137
Bolivia     0      5550        274           0.0494
            1      5574        267           0.0479
Chile       0      9853        474           0.0481
            1      9884        507           0.0513
Colombia    0     27088       1411           0.0521
            1     26972       1364           0.0506
Costa Rica  0      2660        139           0.0523
            1      2649        145           0.0547
Ecuador     0      8036        395           0.0492
            1      7859        385           0.0490
El Salvador 0      4108        220           0.0536
            1      4067        195           0.0479
Guatemala   0      7622        386           0.0506
            1      7503        365           0.0486
Honduras    0      4361        222           0.0509
            

In [17]:
# Exclude Spain
df_no_spain = df[df['country'] != 'Spain']

# Recalculate conversion rates
df_no_spain.groupby('test').agg(
    users=('user_id', 'count'),
    converted=('conversion', 'sum'),
    conversion_rate=('conversion', 'mean')
).round(4)

,users,converted,conversion_rate
test,,,
0,185311,8949,0.0483
1,215774,9367,0.0434


In [18]:
from statsmodels.stats.proportion import proportions_ztest

# Get the new counts from the output above
control_users = df_no_spain[df_no_spain['test'] == 0]
test_users = df_no_spain[df_no_spain['test'] == 1]

conversions = [control_users['conversion'].sum(), test_users['conversion'].sum()]
nobs = [len(control_users), len(test_users)]

stat, p_value = proportions_ztest(conversions, nobs)
print(f'z-statistic: {stat:.4f}')
print(f'p-value: {p_value:.4f}')

z-statistic: 7.3818
p-value: 0.0000


In [19]:
split_check = df_no_spain.groupby(['country', 'test'])['user_id'].count().unstack(fill_value=0)
split_check.columns = ['control_n', 'test_n']
split_check['ratio'] = (split_check['test_n'] / split_check['control_n']).round(3)

print(split_check.sort_values('ratio'))


             control_n  test_n  ratio
country                              
Honduras          4361    4207  0.965
Nicaragua         3419    3304  0.966
Ecuador           8036    7859  0.978
Guatemala         7622    7503  0.984
Venezuela        16149   15905  0.985
El Salvador       4108    4067  0.990
Costa Rica        2660    2649  0.996
Colombia         27088   26972  0.996
Peru             16869   16797  0.996
Mexico           64209   64275  1.001
Chile             9853    9884  1.003
Bolivia           5550    5574  1.004
Panama            1966    1985  1.010
Paraguay          3650    3697  1.013
Argentina         9356   37377  3.995
Uruguay            415    3719  8.961


In [20]:
# Exclude Spain, Argentina, and Uruguay — the three problem countries
exclude = ['Spain', 'Argentina', 'Uruguay']
df_clean = df[~df['country'].isin(exclude)]

control_clean = df_clean[df_clean['test'] == 0]
test_clean = df_clean[df_clean['test'] == 1]

conversions = [control_clean['conversion'].sum(), test_clean['conversion'].sum()]
nobs = [len(control_clean), len(test_clean)]

stat, p_value = proportions_ztest(conversions, nobs)
print(f'z-statistic: {stat:.4f}')
print(f'p-value: {p_value:.4f}')

z-statistic: -0.3583
p-value: 0.7201
